# Task 2 – Exploratory Data Analysis (EDA)
**Agent:** Claude (claude-sonnet-4-6)  
**Date:** 2026-03-15  
**Dataset:** `data/processed/Claude_clean.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Paths are relative to the project root; adjust if running from notebooks/task2/
FIGURES_DIR = "../../figures/Claude"
os.makedirs(FIGURES_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIG_COUNTER = [0]

def save_and_show(fig, name):
    """Save figure to figures/Claude/ with sequential numbering, then display."""
    FIG_COUNTER[0] += 1
    filename = f"{FIGURES_DIR}/fig_{FIG_COUNTER[0]:02d}_{name}.png"
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    print(f"Saved: {filename}")
    plt.show()
    plt.close(fig)

## 1. Load Dataset

In [ ]:
df = pd.read_csv("../../data/processed/Claude_clean.csv")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Binary target for correlations
df["satisfied"] = (df["satisfaction"] == "satisfied").astype(int)

CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
SERVICE_COLS = [
    "Inflight wifi service", "Departure/Arrival time convenient",
    "Ease of Online booking", "Gate location", "Food and drink",
    "Online boarding", "Seat comfort", "Inflight entertainment",
    "On-board service", "Leg room service", "Baggage handling",
    "Checkin service", "Inflight service", "Cleanliness"
]

## 2. Structure Overview

In [ ]:
print("--- Categorical value counts ---")
for col in CAT_COLS + ["satisfaction"]:
    print(f"\n{col}:")
    display(df[col].value_counts().to_frame())

In [ ]:
print("--- Numeric summary ---")
display(df.describe().T)

## 3. Visualisations

### Fig 1 – Target class distribution

In [ ]:
counts = df["satisfaction"].value_counts()
pcts = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values,
              color=["#4C72B0", "#DD8452"], edgecolor="white", linewidth=0.8)
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 600, f"{pct:.1f}%",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Fig 1 – Satisfaction Class Distribution", fontweight="bold")
ax.set_ylabel("Passenger Count")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
sns.despine()
save_and_show(fig, "satisfaction_distribution")

### Fig 2 – Satisfaction rate by categorical features

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, col in zip(axes, CAT_COLS):
    sat_rate = df.groupby(col)["satisfied"].mean().sort_values(ascending=False) * 100
    colors = sns.color_palette("muted", len(sat_rate))
    bars = ax.barh(sat_rate.index, sat_rate.values, color=colors, edgecolor="white")
    ax.set_xlim(0, 100)
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Satisfaction Rate (%)")
    for bar, val in zip(bars, sat_rate.values):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=9)
axes[0].set_ylabel("Category")
fig.suptitle("Fig 2 – Satisfaction Rate by Categorical Features", fontweight="bold", y=1.01)
plt.tight_layout()
save_and_show(fig, "satisfaction_by_categorical")

### Fig 3 – Age distribution by satisfaction

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, grp in df.groupby("satisfaction"):
    ax.hist(grp["Age"], bins=30, alpha=0.6, label=label, edgecolor="white", linewidth=0.5)
ax.set_title("Fig 3 – Age Distribution by Satisfaction", fontweight="bold")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.legend(title="Satisfaction")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
sns.despine()
save_and_show(fig, "age_distribution_by_satisfaction")

### Fig 4 – Flight distance distribution by satisfaction

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in zip(["satisfied", "neutral or dissatisfied"], ["#4C72B0", "#DD8452"]):
    grp = df[df["satisfaction"] == label]["Flight Distance"]
    ax.hist(grp, bins=40, alpha=0.6, label=label, color=color, edgecolor="white", linewidth=0.4)
ax.set_title("Fig 4 – Flight Distance Distribution by Satisfaction", fontweight="bold")
ax.set_xlabel("Flight Distance (miles)")
ax.set_ylabel("Count")
ax.legend(title="Satisfaction")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
sns.despine()
save_and_show(fig, "flight_distance_by_satisfaction")

### Fig 5 – Mean service ratings by satisfaction group

In [ ]:
service_means = df.groupby("satisfaction")[SERVICE_COLS].mean().T
service_means.columns = ["neutral or dissatisfied", "satisfied"]
service_means["diff"] = service_means["satisfied"] - service_means["neutral or dissatisfied"]
service_means = service_means.sort_values("diff", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(service_means))
width = 0.38
ax.bar(x - width / 2, service_means["satisfied"], width,
       label="Satisfied", color="#4C72B0", edgecolor="white")
ax.bar(x + width / 2, service_means["neutral or dissatisfied"], width,
       label="Neutral/Dissatisfied", color="#DD8452", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(service_means.index, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Mean Rating (0–5)")
ax.set_title("Fig 5 – Mean Service Ratings by Satisfaction Group", fontweight="bold")
ax.legend()
sns.despine()
plt.tight_layout()
save_and_show(fig, "service_ratings_by_satisfaction")

### Fig 6 – Correlation heatmap

In [ ]:
corr_cols = SERVICE_COLS + ["Age", "Flight Distance",
                             "Departure Delay in Minutes",
                             "Arrival Delay in Minutes", "satisfied"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.4,
            annot_kws={"size": 7}, ax=ax)
ax.set_title("Fig 6 – Correlation Heatmap (Service Ratings & Target)", fontweight="bold", pad=12)
plt.tight_layout()
save_and_show(fig, "correlation_heatmap")

### Fig 7 – Feature correlations with satisfaction

In [ ]:
numeric_cols = SERVICE_COLS + ["Age", "Flight Distance",
                                "Departure Delay in Minutes",
                                "Arrival Delay in Minutes"]
corr_with_target = (
    df[numeric_cols + ["satisfied"]]
    .corr()["satisfied"]
    .drop("satisfied")
    .sort_values(key=abs, ascending=False)
)

colors = ["#4C72B0" if v > 0 else "#DD8452" for v in corr_with_target.values]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(corr_with_target.index[::-1], corr_with_target.values[::-1],
               color=colors[::-1], edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson Correlation with Satisfaction")
ax.set_title("Fig 7 – Feature Correlations with Passenger Satisfaction", fontweight="bold")
for bar, val in zip(bars, corr_with_target.values[::-1]):
    ax.text(val + (0.005 if val >= 0 else -0.005),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center",
            ha="left" if val >= 0 else "right", fontsize=8)
sns.despine()
plt.tight_layout()
save_and_show(fig, "feature_correlations_with_satisfaction")

## 4. EDA Summary

In [ ]:
print("=== EDA SUMMARY ===")
print(f"Class balance: {counts['satisfied']:,} satisfied ({pcts['satisfied']:.1f}%) | "
      f"{counts['neutral or dissatisfied']:,} dissatisfied ({pcts['neutral or dissatisfied']:.1f}%)")
print("\nTop 5 features positively correlated with satisfaction:")
display(corr_with_target[corr_with_target > 0].head(5).to_frame("correlation"))
print("\nTop 5 features negatively correlated with satisfaction:")
display(corr_with_target[corr_with_target < 0].head(5).to_frame("correlation"))